In [0]:
# =============================================================
# SMOKE TEST 1 — THREE BORROWERS (invoke)
#
# Covers the STANDARD and ENHANCED routes.
# Asserts route, evidence presence, page citations, and output
# contract fields on all three representative borrowers.
# =============================================================

TEST_CASES = [
    {"borrower_id": "B000187", "expected_route": "STANDARD_ANALYST_REVIEW"},
    {"borrower_id": "B001218", "expected_route": "ENHANCED_ANALYST_REVIEW"},
    {"borrower_id": "B005638", "expected_route": "ENHANCED_ANALYST_REVIEW"},
]

TEST_QUESTION = (
    "Assess profitability, leverage and liquidity. "
    "Find evidence for net profit, EBIT, liabilities, "
    "working capital, current assets and equity."
)

for test_case in TEST_CASES:
    graph_output = credit_risk_graph.invoke({
        "borrower_id": test_case["borrower_id"],
        "question":    TEST_QUESTION,
    })

    result       = graph_output["final_result"]
    actual_route = result["policy_assessment"]["workflow_route"]

    assert actual_route == test_case["expected_route"]
    assert result["human_review_required"]      is True
    assert result["automatic_lending_decision"] is False
    assert len(result["document_evidence"]) > 0
    assert all(
        e["page_number"] is not None
        for e in result["document_evidence"]
    )

    print("=" * 75)
    print(f"Borrower    : {result['borrower_id']}")
    print(f"Probability : {result['model_assessment']['probability_of_bankruptcy']:.4f}")
    print(f"Route       : {actual_route}")
    print(f"Rules       : {result['policy_assessment']['triggered_rule_count']}")
    print(f"Evidence    : {len(result['document_evidence'])} chunks")
    for rule in result["policy_assessment"]["triggered_rules"]:
        print(f"  - {rule['rule_id']} [{rule['severity']}]: {rule['message']}")

print("\nSmoke test 1 passed: STANDARD and ENHANCED routes.")

In [0]:
# =============================================================
# SMOKE TEST 2 — DOCUMENT-COMPLETION PATH (stream)
#
# Verifies the architectural gate: when required documents are
# missing, the graph must route to DOCUMENT_COMPLETION_REQUIRED
# and must NOT execute retrieve_evidence.
#
# stream_mode="updates" exposes the exact node execution path.
# No source files or Delta tables are changed.
# =============================================================

original_manifest = document_manifest_df.copy()

try:
    # Simulate a missing financial_summary for B000187.
    document_manifest_df = original_manifest[
        ~(
            (original_manifest["borrower_id"]   == "B000187")
            & (original_manifest["document_type"] == "financial_summary")
        )
    ].copy()

    executed_nodes: list[str] = []
    blocking_result = None

    for update in credit_risk_graph.stream(
        {"borrower_id": "B000187", "question": TEST_QUESTION},
        stream_mode="updates",
    ):
        assert isinstance(update, dict)
        executed_nodes.extend(update.keys())
        if "finalize_case" in update:
            blocking_result = update["finalize_case"]["final_result"]

    assert blocking_result is not None, "Graph did not produce a final result."

    route             = blocking_result["policy_assessment"]["workflow_route"]
    missing_documents = blocking_result["policy_assessment"]["missing_document_types"]
    evidence          = blocking_result["document_evidence"]

    # Validate blocking route
    assert route == "DOCUMENT_COMPLETION_REQUIRED"
    assert "financial_summary" in missing_documents

    # Verify exact execution path
    assert "validate_request"    in executed_nodes
    assert "apply_policy"        in executed_nodes
    assert "document_completion" in executed_nodes
    assert "finalize_case"       in executed_nodes

    # Critical architectural assertion
    assert "retrieve_evidence" not in executed_nodes, (
        "Evidence retrieval should be skipped when required documents are missing."
    )
    assert "enhanced_review" not in executed_nodes
    assert "standard_review" not in executed_nodes

    # Finalize contract on this path
    assert evidence == []
    assert blocking_result["human_review_required"]      is True
    assert blocking_result["automatic_lending_decision"] is False

    print("=" * 75)
    print("Smoke test 2 passed: DOCUMENT_COMPLETION_REQUIRED path")
    print(f"Route          : {route}")
    print(f"Missing docs   : {missing_documents}")
    print(f"Evidence       : {len(evidence)} chunks")
    print(f"Executed nodes : {executed_nodes}")

finally:
    document_manifest_df = original_manifest